# H13 — 更新失败，设备还能回来

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H13 — 更新失败，设备还能回来。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H13-ota.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

桌面工具已交付，修一个问题不应总拆开接 USB。但新固件无法启动时必须有旧版本和串口救援。

前置：H11/H12；可恢复的 4 MB+ 板、可信 HTTPS 固件服务。

## 本次够用的知识

OTA 需要合适分区、候选镜像、bootloader 状态与启动确认。TLS 校验证书/服务器身份，镜像校验验证完整性，签名验证软件来源，三者不等价。pending verify 不能一启动就无条件确认。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 先备份和检查 Flash 空间 · Guided

读 ../OTA.md，确认 Flash 至少 4 MB、当前配置数据已备份、串口可救援。默认单 app 分区不能直接实现双槽 OTA。
修改分区会改变存储布局；先保存当前固件/分区/配置和明确恢复方式。不要对唯一正在使用的设备直接测试失败更新。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 给两个候选槽留空间 · Guided

使用本课 partitions.csv：nvs、otadata、phy_init、ota_0、ota_1。menuconfig 选择 custom table、启用 CONFIG_BOOTLOADER_APP_ROLLBACK_ENABLE、CONFIG_MBEDTLS_CERTIFICATE_BUNDLE。
检查二进制大小不超过槽容量，idf.py partition-table 后阅读真实地址。分区/bootloader 配置首次需要串口烧录完整工程，OTA app 更新不会替你更新 bootloader。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## HTTPS 拉取候选固件 · Guided

REQUIRES esp_https_ota app_update esp_http_client mbedtls，加 H07 网络 helper。URL 是你可控制的 HTTPS 固件服务器，example.invalid 初始值故意不可用；不要禁用证书校验让演示“成功”。
先构建 0.1.0 并串口烧录，再构建 0.2.0 上传 app .bin（不是合并镜像）；安装候选后手动复位观察版本与分区。示例的健康检查只测 IP，正式工具由你补完整条件。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H13/examples/ota.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_https_ota app_update esp_http_client mbedtls esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "wifi_link.hpp"
#include "esp_crt_bundle.h"
#include "esp_https_ota.h"
#include "esp_ota_ops.h"
#include "esp_log.h"
#include "sdkconfig.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    ESP_ERROR_CHECK(wifi_link_start());
    while (!wifi_link_wait(pdMS_TO_TICKS(1000))) {
    }
    const esp_partition_t* running = esp_ota_get_running_partition();
    esp_ota_img_states_t state;
    if (esp_ota_get_state_partition(running, &state) == ESP_OK &&
        state == ESP_OTA_IMG_PENDING_VERIFY) {
        // Demo health test: obtaining IP. Challenge replaces this with product health criteria.
        ESP_ERROR_CHECK(esp_ota_mark_app_valid_cancel_rollback());
        ESP_LOGI("ota", "candidate confirmed by demo IP check");
    }
    ESP_LOGI("ota", "running=%s version=%s", running->label,
             esp_app_get_description()->version);
    esp_http_client_config_t http{};
    http.url = CONFIG_ACADEMY_OTA_URL;
    http.crt_bundle_attach = esp_crt_bundle_attach;
    http.timeout_ms = 15000;
    esp_https_ota_config_t ota{};
    ota.http_config = &http;
    const esp_err_t result = esp_https_ota(&ota);
    ESP_LOGI("ota", "update result=%s", esp_err_to_name(result));
    // Do not reboot into an endless update loop. Inspect result, then manually reset to test candidate.
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 健康不是启动一行日志 · Modify

定义候选启动健康：关键外设、Controller、自检、配置版本与必要服务在限定时间内正常。网络可离线的产品不能把 AP 不可用当成固件损坏。
显示 running partition/version、pending/valid 状态。确认健康后 mark_app_valid_cancel_rollback；不健康调用 mark_app_invalid_rollback_and_reboot，或按测试计划复位验证 bootloader 回滚。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 故意让候选不通过自检 · Debug

在候选 0.2.0 加一个软件自检失败开关，不破坏硬件。首次 boot 到 pending verify 后，在确认之前复位，观察旧槽恢复；再使用真实失败判定回滚。
测试下载失败或截断：不能将失败镜像设启动。实验保留串口和稳定旧版，记录实际启动分区，不以 HTTPS 200 就判更新成功。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 有边界的更新状态机 · Build

实现 idle/checking/downloading/ready/pending/valid/rollback，更新由用户授权或明确策略触发，不能每次启动无限下载相同镜像。
检查版本/目标/大小、TLS、镜像和适用的签名策略。断下载、错误 URL、失败自检和空间不足都有记录；保存 config 兼容与恢复说明。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 交付更新手册 · Open

写成功更新与失败回滚两条可复现路径，注明 HTTPS 服务、证书、Flash/分区、构建版本和串口 rescue。未实测项保持未确认。
H14 将在干净目录复现你的持续工程；OTA 只在产品需要远程更新时作为交付要求。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 真实分区与 bin 大小核对，串口救援可用
- [ ] 真实候选更新、启动确认与失败回滚分别记录
- [ ] TLS/完整性/签名边界、配置兼容和下载失败明确

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H13',
    [
        '真实分区与 bin 大小核对，串口救援可用',
        '真实候选更新、启动确认与失败回滚分别记录',
        'TLS/完整性/签名边界、配置兼容和下载失败明确',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H13/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/system/ota.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/system/ota.html)
- [api-reference/system/esp_https_ota.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/system/esp_https_ota.html)
- [api-guides/partition-tables.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-guides/partition-tables.html)